# LBT Environmental Telemetry

Plot data from the LBT and SMT weather stations from a night of observing.

Data are in `/lbt/data/telemetry/tcs/env/CCYY/MM/DD/*.h5` on the LBT summit machines. There are two
telemetry files
 * `CCYYMMDD0001.env.lbt_weather.h5` - LBT weather station telemetry
 * `CCYYMMDD0001.env.smt_weather.h5` - SMT weather station telemetry (includes PWV and $\tau$)
 
We make a 4-panel plot, vertically from top to bottom: 
 * LBT ambient air temperature in degrees C
 * LBT relative humidity with closure limit in %
 * LBT wind speed from the building front and rear anemometers, showing the rapid-cadence data with a 5-min running average to represent "sustained" speed along with the closure limit.
 * Air pressure in hectopascals (hPa)
 
Additional options are to plot the dewpoint temperature in the panel with the air temperature, and to plot
the estimate precipitable water vapor in millimeters from the SMT in an optional 5th bottom panel.  The PWV
data are noisy, so care is needed when interpreting it, but it can provide guidance if using some codes
for creating synthetic telluric absorption spectra.

### Author

Rick Pogge, OSU Astronomy (pogge.1@osu.edu)

Updated: 2024 March 14

In [ ]:
%matplotlib inline

import math
import h5py as hdf
import numpy as np
import glob

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, LogLocator, NullFormatter

# astropy time 

from astropy.time import Time

# scipy uniform_filter1d() method for running means

from scipy.ndimage import uniform_filter1d

# throttle nuisance warnings

import warnings
warnings.filterwarnings('ignore',category=UserWarning, append=True)
warnings.filterwarnings('ignore',category=RuntimeWarning, append=True)

# Standard plot setup

# Plot width and height in pixels

plotHeight = 4000
plotWidth = 4000

# Font and line weight defaults for axes

lwidth = 0.5
matplotlib.rcParams.update({'font.size':8})
matplotlib.rc('axes',linewidth=lwidth)

# LaTeX will be used throughout for markup of symbols

plt.rc('text', usetex=True)
plt.rc('font', **{'family':'serif','serif':['Times-Roman'],'weight':'bold','size':'8'})
plt.rcParams['xtick.major.pad']='5'
plt.rcParams['ytick.major.pad']='5'
plt.rcParams['axes.labelpad'] = '5'

# plot resolution and window

dpi = 600
wDisp = plotWidth
hDisp = plotHeight
wInches = float(wDisp)/float(dpi)
hInches = float(hDisp)/float(dpi)

## Plot LBT and SMT environmental sensor data

Open/sniff/close HDF5 files with the LBT and SMT weather telemetry...

### Open and read the HDF5 telemetry files

We use these data elements from the LBT environmental telemetry:
 * `time_stamp` - MJD in integer microseconds
 * `temperature` - temperature in degrees C
 * `pressure` - air pressure in hPa
 * `humidity` - relative humidity in %
 * `windspeed` - rear anemometer windspeed in m/s
 * `winddirection` - rear anemometer wind true direction in degrees
 * `windspeed_front` - front anemometer windspeed in m/s
 * `winddirection_front` - front anemometer wind true direction in degrees
 * `sky_brightness` - sky brightness in magnitudes
 
We use these data from the SMT environmental telemetry:
 * `time_stamp` - MJD in integer microseconds
 * `amb_temp` - ambient temperature in degrees C
 * `amb_pres` - ambient pressure in hPa
 * `humidity` - relative humidity in %
 * `mm_h2o` - precipitable water vapor column in mm
 * `tau_0` - zenith optical depth a XXX mm

More data later?

In [ ]:
obsDate = '20260524'

# options

hardCopy = False

nightOnly = True
startUTC = 0.0
endUTC = 14.0

# plot dew point and/or precipitable water vapor?

plotDewPt = False
plotPWV = False

# end options

lbtList = glob.glob(f'Env/{obsDate}*.env.lbt_weather.h5')
if len(lbtList)==1:
    lbtFile = lbtList[0]
else:
    lbtFile = 'lbtEnv.h5'
    
smtList = glob.glob(f'Env/{obsDate}*.env.smt_weather.h5')
if len(smtList)==1:
    smtFile = smtList[0]
else:
    smtFile = 'smtEnv.h5'

# LBT Evironmental Data

lbtEnv = hdf.File(lbtFile,'r') # read only
lbtDS = list(lbtEnv.keys())[0]

lbtData = lbtEnv[lbtDS]
#print(lbtData.dtype) # uncomment to see all data values

#print("\nLBT Env Data:")
#
#for datum in ['time_stamp','temperature','humidity','pressure','windspeed','winddirection','sky_brightness',
#             'sky_brightness_timestamp']:
#    print(f'\n{datum}:')
#    for key in ['Description','Units']:
#        print(f"  {key}: {lbtData.attrs[key][datum]}")

lbtMJD = lbtData['time_stamp']*1.0e-6/86400.0 # convert MJD in microseconds to days
lbtMJD0 = int(lbtMJD[0])
lbtHour = 24.0*(lbtMJD-lbtMJD0)
      
t = Time([lbtMJD0], format='mjd', scale='utc')
utcDate = f"{t.to_value('iso',subfmt='date')[0]}"

lbtTemp = lbtData['temperature']
lbtDewP = lbtData['dewpoint']
lbtPres = lbtData['pressure']
lbtRH = lbtData['humidity']
lbtWSf = lbtData['windspeed_front']
lbtWDf = lbtData['winddirection_front']
lbtWSr = lbtData['windspeed']
lbtWDr = lbtData['winddirection']
lbtSky = lbtData['sky_brightness']

lbtEnv.close()

# SMT Environmental Data

smtEnv = hdf.File(smtFile,'r')
smtDS = list(smtEnv.keys())[0]

#print("\nSMT Env Data:")

smtData = smtEnv[smtDS]
#print(smtData.dtype) # uncomment to see all data values

#for datum in ['time_stamp','amb_temp','amb_pres','humidity','mm_h2o','tau_0','tauz','tausigma','refract','tipper_az']:
#    print(f'\n{datum}:')
#    for key in ['Description','Units']:
#        print(f"  {key}: {smtData.attrs[key][datum]}")
        
smtMJD = smtData['time_stamp']*1.0e-6/86400.0 # convert MJD in microseconds to days
smtMJD0 = int(smtMJD[0])
smtHour = 24.0*(smtMJD-smtMJD0)

smtTemp = smtData['amb_temp']
smtPres = smtData['amb_pres']
smtRH = smtData['humidity']
smtPWV = smtData['mm_h2o']
tau0 = smtData['tau_0']

smtEnv.close()

i = -1

print("Last Measurement:")
t = Time(lbtMJD[i], format='mjd', scale='utc')
print(f"  LBT: {t.to_value('iso')} {lbtTemp[i]:.1f} C {lbtPres[i]:.1f} hPa {lbtRH[i]:.1f}% front={lbtWSf[i]:.1f} m/s rear={lbtWSr[i]:.1f} m/s Sky={lbtSky[i]:.1f} mag")
    
t = Time(smtMJD[i], format='mjd', scale='utc')
print(f"  SMT: {t.to_value('iso')} {smtTemp[i]:.1f} C {smtPres[i]:.1f} hPa {smtRH[i]:.1f}% PWV={smtPWV[i]:.2f}mm tau0={tau0[i]:.2f}")

# plotting limits

# UTC time

minTime = np.min([np.min(lbtHour),np.min(smtHour)])
maxTime = np.max([np.max(lbtHour),np.max(smtHour)])
dT = maxTime - minTime
tMin = minTime # np.floor(minTime) # minTime - 0.05*dT
tMax = maxTime # np.ceil(maxTime+0.01*dT) # maxTime + 0.05*dT

tMin = 0.0
if nightOnly and tMax > endUTC:
     tMax = endUTC

# temperature

if plotDewPt:
    minTemp = np.min([np.min(lbtTemp),np.min(lbtDewP)])
else:
    minTemp = np.min(lbtTemp)
    
maxTemp = np.max(lbtTemp)
dTemp = maxTemp - minTemp
minTemp -= 0.1*dTemp
maxTemp += 0.1*dTemp

# pressure

presFloor = 600.0 # hPa floor, less than this is probably missing data

minPres = np.min(lbtPres[np.where(lbtPres>presFloor)])
maxPres = np.max(lbtPres[np.where(lbtPres>presFloor)])
medPres = np.median(lbtPres[np.where(lbtPres>presFloor)])
dP = maxPres - minPres
if dP < 10:
    minPres = medPres - 5.0
    maxPres = medPres + 5.0
else:
    minPres -= 0.05*dP
    maxPres += 0.05*dP
#minPres = 670
#maxPres = 690

# humidity

minRH = np.min(lbtRH)
maxRH = np.max(lbtRH)
minRH = 0
maxRH = 105

# wind speeds

minWS = 0.0 # np.min([np.min(lbtWSf),np.min(lbtWSr)])
maxWS = 1.05* np.max([np.max(lbtWSf),np.max(lbtWSr)])
wsLimit = 20.0 # m/s

if maxWS < wsLimit:
    maxWS = wsLimit+5.0
elif maxWS > 35:
    maxWS = 35.0
    
# PWV

minPWV = 0.0 # np.min(smtPWV)
maxPWV = 1.05*np.max(smtPWV)

# running mean of the wind speed.  Sampling is ~1 second so nFilt=60 is about 1 minute rolling mean

nFilt = 300 

smWSf = uniform_filter1d(lbtWSf, size=nFilt)
smWDf = uniform_filter1d(lbtWDf, size=nFilt)

smWSr = uniform_filter1d(lbtWSr, size=nFilt)
smWDr = uniform_filter1d(lbtWDr, size=nFilt)

## Windrose plot

Plot wind speed and direction in polar "windrose" plots where radius is speed and angle is compass direction
of the wind.

Need to convert angle in degrees to radians.

Plot transparent points.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':'polar'},figsize=(wInches,wInches),dpi=dpi)

ax.plot(0,0,'o',ms=2,mfc='green',mec=None,mew=0,zorder=1,label='Front')
ax.plot(0,0,'o',ms=2,mfc='blue',mec=None,mew=0,zorder=1,label='Rear')

WSf = lbtWSf
WDf = lbtWDf
WSr = lbtWSr
WDr = lbtWDr

alpha = 0.08/(len(WSf)/1.0e4)
if alpha < 0.01:
    alpha = 0.01

ax.plot(np.radians(WDf),WSf,'o',ms=2,mfc='green',mec=None,mew=0,alpha=alpha,zorder=8)
iMax = np.where(WSf > 20.0)[0]
if len(iMax) > 0:
    ax.plot(np.radians(WDf[iMax]),WSf[iMax],'o',ms=1,mfc='red',mec=None,mew=0,alpha=2*alpha,zorder=10)

ax.plot(np.radians(WDr),WSr,'o',ms=2,mfc='blue',mec=None,mew=0,alpha=alpha,zorder=9)
iMax = np.where(WSr > 20.0)[0]
if len(iMax) > 0:
    ax.plot(np.radians(WDr[iMax]),WSr[iMax],'o',ms=2,mfc='red',mec=None,mew=0,alpha=2*alpha,zorder=10)
    
ax.set_rmax(maxWS)
ax.set_rticks([5,10,15,20,25,30])

ax.set_thetagrids([0,45,90,135,180,225,270,315],['E','SE','N','NW','W','SW','S','SE'])

ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
ax.grid(True,lw=0.5)
ax.legend(fontsize=6,ncol=4,markerscale=2)

# maximum wind speed limit

ax.plot(np.linspace(0,2.0*np.pi,301),20.0*np.ones(301),'--',lw=0.8,zorder=10,color='#bb0000')

labPA = ax.get_rlabel_position()
ax.text(np.radians(labPA),ax.get_rmax()+1,'wind\nspeed\n[m/s]',rotation=labPA,ha='left',va='center')

ax.set_title(rf'LBT wind speed and direction {utcDate} UTC', va='bottom')

# hardcopy

if hardCopy:
    plotFile = f'{obsDate}_wind.png'
    plt.savefig(plotFile) # ,bbox_inches='tight')#,facecolor='white')

## Polar histogram version



In [ ]:
# tweak the histogram scaling colormap - larger vScale = less contrast

vScale = 2.0

# plot

fig, ax = plt.subplots(subplot_kw={'projection':'polar'},figsize=(wInches,wInches),dpi=dpi)

ax.plot(0,0,'o',ms=2,mfc='green',mec=None,mew=0,zorder=1,label='Front')
ax.plot(0,0,'o',ms=2,mfc='blue',mec=None,mew=0,zorder=1,label='Rear')

WSf = lbtWSf
WDf = lbtWDf
WSr = lbtWSr
WDr = lbtWDr

# concatenate both speed and direction data in a single vector in appropriate units

WS = np.concatenate((WSf,WSr)) # wind speed for both sensors
WD = np.concatenate((np.radians(WDf),np.radians(WDr))) # wind direction in radians for both sensors

# define binning

rbins = np.linspace(0,int(maxWS),1+2*int(maxWS))   # 0.5 m/s speed bins
abins = np.linspace(0,2*np.pi,73) # 5 degree azimuth bins

# build the histograms

hist, _, _ = np.histogram2d(WD, WS, bins=(abins, rbins),density=True)
A, R = np.meshgrid(abins, rbins)

# max display to avoid single bin saturation of the color map

numBins =len(rbins)+len(abins)
vmax = vScale*8.0/numBins

# plot

cmap = 'gist_heat_r'

pc = ax.pcolormesh(A, R, hist.T, cmap=cmap, vmax=vmax)
#fig.colorbar(pc)

# custom axes

ax.set_rmax(maxWS)
ax.set_rticks([5,10,15,20,25,30])

ax.set_thetagrids([0,45,90,135,180,225,270,315],['E','SE','N','NW','W','SW','S','SE'])

ax.set_rlabel_position(-22.5)  # Move radial labels away from plotted line
ax.grid(True,lw=0.5)

# maximum wind speed limit

ax.plot(np.linspace(0,2.0*np.pi,301),20.0*np.ones(301),'--',lw=0.8,zorder=10,color='#bb0000')

labPA = ax.get_rlabel_position()
ax.text(np.radians(labPA),ax.get_rmax()+1,'wind\nspeed\n[m/s]',rotation=labPA,ha='left',va='center')

ax.set_title(rf'LBT wind speed and direction {utcDate} UTC', va='bottom')

# show it

plt.show()

# hardcopy

if hardCopy:
    plotFile = f'{obsDate}_windrose.png'
    plt.savefig(plotFile) # ,bbox_inches='tight')#,facecolor='white')